# 01 - EDA Current Stress
Notebook EDA dengan tracking MLflow yang konsisten.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import sys

CANDIDATE_DIRS = [
    Path.cwd(),
    Path.cwd() / "nostressia-machine-learning" / "Current-Stress" / "notebooks" / "experiments",
]
for _dir in CANDIDATE_DIRS:
    if (_dir / "mlflow_utils.py").exists():
        sys.path.insert(0, str(_dir))
        break


import mlflow
import pandas as pd
import matplotlib.pyplot as plt

from mlflow_utils import configure_mlflow, set_seeds, RANDOM_STATE, load_current_stress_dataset, log_run_metadata, temp_artifact_dir

repo_root = configure_mlflow()
set_seeds(RANDOM_STATE)

In [ ]:
raw_df, feature_df, y = load_current_stress_dataset(repo_root)

with mlflow.start_run(run_name="EDA - Current Stress") as run:
    log_run_metadata(
        run_description="EDA baseline for current stress dataset; dataset=current_stress_v1; split=80/20 metadata only",
        tags={"features": "all", "model": "EDA"},
        params={"stage": "eda", "rows": len(raw_df), "columns": raw_df.shape[1]},
        dataset_df=raw_df.assign(target=y.values),
        dataset_context="eda",
    )

    with temp_artifact_dir() as td:
        art = Path(td)
        raw_df.describe(include="all").transpose().to_csv(art / "eda_describe.csv")
        raw_df.isna().sum().rename("missing_count").to_csv(art / "missing_values.csv")

        fig, ax = plt.subplots(figsize=(6, 4))
        raw_df["Stress_Level"].value_counts().sort_index().plot(kind="bar", ax=ax, color="#4C72B0")
        ax.set_title("Stress Label Distribution")
        ax.set_xlabel("Stress Level")
        ax.set_ylabel("Count")
        fig.tight_layout()
        fig.savefig(art / "stress_label_distribution.png", dpi=160)
        plt.close(fig)

        mlflow.log_artifacts(str(art), artifact_path="eda")

    print("EDA run ID:", run.info.run_id)